In [63]:
from pathlib import Path

import numpy as np
import pandas as pd
from beliefppg import infer_hr
import os

from sklearn.metrics import mean_absolute_error

In [64]:
BASE_PATH = Path("../../data/processed/dalia")
PATIENTS = [f"S{i}" for i in range(1, 16)]  # S1 to S15


def load_bvp_acc_hr(patient: str):
    patient_path = os.path.join(BASE_PATH, patient)
    bvp = pd.read_csv(os.path.join(patient_path, "wrist/wrist_BVP.csv"))
    acc = pd.read_csv(os.path.join(patient_path, "wrist/wrist_ACC.csv"))
    label = pd.read_csv(os.path.join(patient_path, "label.csv"))

    return bvp, acc, label

In [65]:
for p in PATIENTS:
    bvp_df, acc_df, hr_df = load_bvp_acc_hr(p)

    print(f"{p} - BVP shape: {bvp_df.shape}, ACC shape: {acc_df.shape}, HR shape: {hr_df.shape}")

    # 1. Convert DataFrames to NumPy arrays
    bvp_arr = bvp_df.values
    acc_arr = acc_df.values

    # 2. Run inference with the correct 32 Hz ACC frequency
    pred_hr, idx = infer_hr(
        ppg=bvp_arr,
        ppg_freq=64,
        acc=acc_arr,
        acc_freq=32  # <-- CRITICAL: Changed from 64 to 32
    )

    print(f"Predicted HR array shape: {pred_hr.shape}")

    # 1. Convert the 'idx' (which is in 64Hz sample numbers) to seconds
    pred_time_sec = idx / 64.0

    # 2. Create a time array for your ground truth HR
    # (4603 values, spaced by 2 seconds. The first 8-second window centers at 4 seconds)
    gt_time_sec = np.arange(len(hr_df)) * 2.0 + 4.0

    # 3. Interpolate the 4597 predictions to match the 4603 ground truth timestamps
    aligned_pred_hr = np.interp(
        gt_time_sec,       # The x-coordinates we want to evaluate at
        pred_time_sec,     # The x-coordinates we have
        pred_hr            # The y-coordinates we have
    )

    # 4. Reshape hr_df to a 1D array if it isn't already
    gt_hr_1d = hr_df.values.flatten()

    # 5. Calculate the final error!
    mae = mean_absolute_error(gt_hr_1d, aligned_pred_hr)

    print(f"Final aligned prediction shape: {aligned_pred_hr.shape}")
    print(f"Ground truth shape: {gt_hr_1d.shape}")
    print(f"Mean Absolute Error (MAE): {mae:.2f} BPM")



S1 - BVP shape: (589568, 1), ACC shape: (294784, 3), HR shape: (4603, 1)
37/37 [==============================] - 5s 116ms/step
Predicted HR array shape: (4597,)
Final aligned prediction shape: (4603,)
Ground truth shape: (4603,)
Mean Absolute Error (MAE): 4.20 BPM
S2 - BVP shape: (525120, 1), ACC shape: (262560, 3), HR shape: (4099, 1)
33/33 [==============================] - 5s 120ms/step
Predicted HR array shape: (4093,)
Final aligned prediction shape: (4099,)
Ground truth shape: (4099,)
Mean Absolute Error (MAE): 4.15 BPM
S3 - BVP shape: (559424, 1), ACC shape: (279712, 3), HR shape: (4367, 1)
36/36 [==============================] - 7s 145ms/step
Predicted HR array shape: (4361,)
Final aligned prediction shape: (4367,)
Ground truth shape: (4367,)
Mean Absolute Error (MAE): 4.48 BPM
S4 - BVP shape: (585600, 1), ACC shape: (292800, 3), HR shape: (4572, 1)
37/37 [==============================] - 6s 143ms/step
Predicted HR array shape: (4566,)
Final aligned prediction shape: (4572,)
